# ARC-AGI-3 GPU 冒烟 — vLLM + Qwen3.6-35B-A3B-FP8

验证三件事: ① wheelhouse 离线装 vLLM ② 起服务加载 FP8 权重(**必须 RTX 6000**, T4/P100 无 FP8) ③ LLM-agent 打一局。
这是开发用 notebook, 不是提交物。


In [ ]:
import os, sys, subprocess, time, json
from pathlib import Path

os.environ["ONLY_RESET_LEVELS"] = "true"
WORKING = Path("/kaggle/working")
import torch
print("GPU:", torch.cuda.get_device_name(0), "| CC:", torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0) >= (8, 9), "FP8 需要 Ada(CC>=8.9), 请把加速器换成 RTX 6000"


## 1. 离线装 vLLM(Duck 公开 wheelhouse) + arc-agi(比赛 wheelhouse)


In [ ]:
def find_wheelhouse(pattern):
    for p in Path("/kaggle/input").rglob(pattern):
        return p.parent
    raise RuntimeError(f"找不到 {pattern}")

vllm_wheels = find_wheelhouse("vllm-*.whl")
print("vllm wheelhouse:", vllm_wheels)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
                       "--no-warn-conflicts", "--find-links", str(vllm_wheels), "vllm"])
arc_wheels = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
                       "--no-warn-conflicts", "--find-links", str(arc_wheels), "arc-agi"])
import importlib.metadata as md_
print("vllm", md_.version("vllm"), "| arc-agi", md_.version("arc-agi"))


## 2. 定位模型与源码 bundle


In [ ]:
model_dir = None
for cfg in Path("/kaggle/input").rglob("config.json"):
    d = cfg.parent
    if list(d.glob("*.safetensors")):
        model_dir = d
        break
assert model_dir, "找不到模型目录(要挂 Kaggle Model)"
print("model:", model_dir)

bundle = next(Path("/kaggle/input").rglob("arc3-jinbo-bundle.json")).parent
sys.path.insert(0, str(bundle))
print("bundle:", bundle)


## 3. 起 vLLM 服务(权重 37GB, 加载可能 10-20 分钟)


In [ ]:
from kaggle_agent.serve_vllm import start_vllm
proc = start_vllm(str(model_dir), port=8000, max_model_len=16384)


## 4. 冒烟对话


In [ ]:
from kaggle_agent.llm import LLMClient
llm = LLMClient("http://127.0.0.1:8000", model="local")
print(llm.chat([{"role": "user", "content": "用一句话解释什么是网格解谜游戏。"}]))
print(vars(llm.stats))


## 5. LLM-agent 打一局(离线, 小预算)


In [ ]:
os.environ["A3_LLM_BASE_URL"] = "http://127.0.0.1:8000"
os.environ["A3_LLM_MODEL"] = "local"
os.environ["A3_AGENT"] = "llm"

from kaggle_agent.run_submission import main
env_dir = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"))
summary = main(env_dir=env_dir, games=["ft09"], seconds_per_game=900, max_actions=300,
               out_dir=str(WORKING))
print(json.dumps({k: v for k, v in summary.items() if k != "scorecard"}, ensure_ascii=False, indent=2))
